In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# preprocessing for downstream experiment 0&1 LDA/NMF
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_tickets.csv'
OUT_MAIN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_input.csv'

In [3]:
import re

df = pd.read_csv(INPUT_PATH)

def extract_description_only(row):
    text  = str(row['combined_text']).strip()
    subj  = str(row['Ticket Subject']).strip()
    ttype = str(row['Ticket Type']).strip()
    if text.startswith(subj):
        text = text[len(subj):].strip()
    if text.endswith(ttype):
        text = text[:-len(ttype)].strip()
    text = re.sub(r'\{[^{}]+\}', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_out = pd.DataFrame({
    'ticket_id':           [f'TICKET_{i}' for i in range(len(df))],
    'description':         df['combined_text'].astype(str).str.strip(),
    'description_only':    df.apply(extract_description_only, axis=1),
    'free_text': (
        df['combined_text']
        .astype(str)
        .str.replace(r'\{[^{}]+\}', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    ),
    'text_cleaned_axis1':  df['clean_text'].astype(str).str.strip(),
    'Ticket Type':         df['Ticket Type'].astype(str).str.strip(),
    'Ticket Subject':      df['Ticket Subject'].astype(str).str.strip(),
    'Ticket Priority':     df['Ticket Priority'].astype(str).str.strip(),
    'text_length':         df['text_length'].astype(int)
})

df_out.head()

,ticket_id,description,description_only,free_text,text_cleaned_axis1,Ticket Type,Ticket Subject,Ticket Priority,text_length
0,TICKET_0,Product setup I'm having an issue with the {pr...,I'm having an issue with the . Please assist. ...,Product setup I'm having an issue with the . P...,setup billing zip code appreciate requested we...,Technical issue,Product setup,Critical,18
1,TICKET_1,Peripheral compatibility I'm having an issue w...,I'm having an issue with the . Please assist. ...,Peripheral compatibility I'm having an issue w...,peripheral compatibility change existing facin...,Technical issue,Peripheral compatibility,Critical,10
2,TICKET_2,Network problem I'm facing a problem with my {...,I'm facing a problem with my . The is not turn...,Network problem I'm facing a problem with my ....,network facing turning fine until now respond ...,Technical issue,Network problem,Low,13
3,TICKET_3,Account access I'm having an issue with the {p...,I'm having an issue with the . Please assist. ...,Account access I'm having an issue with the . ...,account access interested love see happen chec...,Billing inquiry,Account access,Low,15
4,TICKET_4,Data loss I'm having an issue with the {produc...,I'm having an issue with the . Please assist. ...,Data loss I'm having an issue with the . Pleas...,data loss note seller responsible damages aris...,Billing inquiry,Data loss,Low,23


In [4]:
df_out = df_out[df_out['text_cleaned_axis1'].ne('')].reset_index(drop=True)

df_out.to_csv(OUT_MAIN_PATH, index=False, encoding='utf-8')